# YOLOv8 — treinamento e retomada automática
Selecione GPU no Colab. Em uma sessão nova, envie o ZIP, instale as dependências e monte o Drive. Depois execute as células de retomada: experimentos concluídos são preservados, incompletos são retomados e os restantes são iniciados.
Não execute o treinamento em duas sessões simultaneamente. Resultados e logs ficam no Drive; os dados extraídos em /content são temporários.


In [ ]:
import zipfile
from pathlib import Path

from google.colab import files

uploaded = files.upload()
assert "colab_bundle.zip" in uploaded, "Selecione colab_bundle.zip"
project = Path("/content/what-is-the-book-2")
project.mkdir(exist_ok=True)
with zipfile.ZipFile("colab_bundle.zip") as archive:
    assert all((project / name).resolve().is_relative_to(project) for name in archive.namelist())
    archive.extractall(project)
del uploaded
%cd /content/what-is-the-book-2

In [ ]:
import os

os.environ["MPLBACKEND"] = "Agg"
import subprocess

subprocess.run(["pip", "install", "uv"], check=True)
subprocess.run(
    ["uv", "sync", "--locked", "--python", "3.13", "--extra", "training", "--no-dev"], check=True
)
subprocess.run(
    [
        "uv",
        "run",
        "--no-sync",
        "python",
        "-c",
        "import torch; print(torch.__version__, torch.version.cuda); "
        "assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))",
    ],
    check=True,
)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
output = "/content/drive/MyDrive/what-is-the-book-2/runs"
base = ["uv", "run", "--no-sync", "python", "src/detection/train_grid.py", "--output", output]
subprocess.run(base, check=True)  # Apenas apresenta as oito combinações

## Verificação curta opcional
Deixe RUN_SMOKE=False para retomar treinamentos. Ative apenas para verificar um ambiente novo. O teste usa uma época e 5% do treino, em uma pasta de diagnóstico separada. A saída completa é salva no Drive, sem inundar o notebook.


In [ ]:
from datetime import datetime
from pathlib import Path

RUN_SMOKE = False
if RUN_SMOKE:
    diagnostic = Path(output) / "diagnosticos" / datetime.now().strftime("%Y%m%d_%H%M%S")
    diagnostic.mkdir(parents=True, exist_ok=False)
    smoke_command = [
        "uv",
        "run",
        "--no-sync",
        "python",
        "-u",
        "src/detection/train_grid.py",
        "--output",
        str(diagnostic),
        "--run",
        "--smoke",
        "--experiment",
        "1",
    ]
    smoke_log = diagnostic / "smoke.log"
    print(f"Teste curto. Log: {smoke_log}", flush=True)
    with smoke_log.open("w", encoding="utf-8") as handle:
        smoke_result = subprocess.run(
            smoke_command,
            stdout=handle,
            stderr=subprocess.STDOUT,
            env={**os.environ, "MPLBACKEND": "Agg", "PYTHONUNBUFFERED": "1"},
        )
    print("\n".join(smoke_log.read_text().splitlines()[-15:]))
    smoke_result.check_returncode()

## Treinamento com retomada automática
Execute as duas células abaixo, sem selecionar o número do experimento. A primeira define o programa; a segunda executa no ambiente isolado.
- result.json compatível: pula o experimento concluído.
- last.pt com estado de treinamento: retoma do checkpoint salvo.
- Treinamento encerrado com best.pt: gera validação e resultado pendentes.
- Experimento inexistente: inicia com as configurações da grade.
- Pasta incompleta sem checkpoint ou metadados incompatíveis: interrompe para inspeção.
Use somente checkpoints produzidos por você. A retomada confere dataset, configuração e versões. O ranking usa apenas validação; o teste permanece reservado.
Após uma desconexão, confira as sessões ativas antes de executar novamente. O notebook mostra uma mensagem por época; logs completos ficam em runs/logs/.


In [ ]:
code = r"""
import hashlib
import itertools
import json
from pathlib import Path

import torch
import ultralytics
from ultralytics import YOLO

root = Path('/content/what-is-the-book-2')
grid = Path('/content/drive/MyDrive/what-is-the-book-2/runs/grid')
grid.mkdir(parents=True, exist_ok=True)
cfg = json.loads((root / 'configs/yolov8_grid.json').read_text())
manifest = (root / 'data/unified/manifest.json').read_bytes()
dataset_hash = hashlib.sha256(manifest).hexdigest()
data = grid / 'dataset.yaml'
data.write_text(
    f"path: {(root / 'data/unified').as_posix()}\n"
    'train: train/images\nval: val/images\ntest: test/images\n'
    'nc: 1\nnames: [book]\n', encoding='utf-8',
)
assert torch.cuda.is_available(), 'Ative a GPU no Colab.'
keys = list(cfg['grid'])
combinations = itertools.product(*(cfg['grid'][key] for key in keys))


def save_json(path, value):
    temporary = path.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(value, indent=2), encoding='utf-8')
    temporary.replace(path)


def update_ranking():
    results = [
        json.loads(path.read_text(encoding='utf-8'))
        for path in grid.glob('exp_*/result.json')
    ]
    results.sort(key=lambda item: item['metrics']['metrics/mAP50-95(B)'], reverse=True)
    save_json(grid / 'ranking.json', results)


for number, values in enumerate(combinations, 1):
    name = f'exp_{number:02d}'
    folder = grid / name
    result_path = folder / 'result.json'
    request_path = grid / f'{name}_request.json'
    settings = {**cfg['fixed'], **dict(zip(keys, values, strict=True))}
    expected = {
        'id': number, 'model': cfg['model'], 'settings': settings,
        'dataset_sha256': dataset_hash,
    }
    if result_path.exists():
        result = json.loads(result_path.read_text(encoding='utf-8'))
        for key, value in expected.items():
            if result.get(key) != value:
                raise RuntimeError(f'{name}: resultado de outro dataset/configuracao.')
        print(f'STATUS | {name}: concluido; pulando.', flush=True)
        continue
    if request_path.exists():
        request = json.loads(request_path.read_text(encoding='utf-8'))
        for key, value in expected.items():
            if request.get(key) != value:
                raise RuntimeError(f'{name}: dataset/configuracao mudou desde a execucao anterior.')
    else:
        if folder.exists():
            raise RuntimeError(f'{name}: pasta existente sem metadados; confira antes de retomar.')
        request = {
            **expected, 'ultralytics': ultralytics.__version__,
            'torch': torch.__version__, 'device': '0',
        }
        save_json(request_path, request)
    if (
        request.get('ultralytics') != ultralytics.__version__
        or request.get('torch') != torch.__version__
    ):
        raise RuntimeError(f'{name}: versoes diferentes do ambiente original.')
    last = folder / 'weights/last.pt'
    best = folder / 'weights/best.pt'
    if last.exists():
        # Load only trusted checkpoints produced by this project.
        checkpoint = torch.load(last, map_location='cpu', weights_only=False)
        epoch = int(checkpoint.get('epoch', -1))
        optimizer = checkpoint.get('optimizer')
        can_resume = epoch >= 0 and optimizer is not None and epoch + 1 < settings['epochs']
        del checkpoint, optimizer
        if can_resume:
            print(f'STATUS | {name}: retomando apos a epoca {epoch + 1}.', flush=True)
            YOLO(str(last)).train(resume=True, data=str(data), device=0)
        elif best.exists():
            print(f'STATUS | {name}: treinamento encerrado; gerando resultado.', flush=True)
        else:
            raise RuntimeError(f'{name}: checkpoint sem estado de retomada e sem best.pt.')
    elif folder.exists():
        raise RuntimeError(f'{name}: pasta sem last.pt; confira o log.')
    else:
        print(f'STATUS | {name}: iniciando treinamento.', flush=True)
        YOLO(cfg['model']).train(
            data=str(data), device=0, project=str(grid), name=name,
            exist_ok=False, **settings,
        )
    assert best.exists(), f'{name}: best.pt nao encontrado.'
    print(f'STATUS | {name}: avaliando na validacao.', flush=True)
    metrics = YOLO(str(best)).val(
        data=str(data), split='val', imgsz=settings['imgsz'], batch=settings['batch'],
        device=0, max_det=1000, project=str(folder), name='validation', plots=True,
    )
    save_json(result_path, {
        **request,
        'metrics': {key: float(value) for key, value in metrics.results_dict.items()},
        'best': str(best),
    })
    update_ranking()
    print(f'STATUS | {name}: concluido e registrado.', flush=True)

update_ranking()
print('STATUS | Grade concluida. O conjunto de teste nao foi avaliado.', flush=True)
"""

In [ ]:
import os
import subprocess
from datetime import datetime
from pathlib import Path

PROJECT = Path("/content/what-is-the-book-2")
OUTPUT = Path("/content/drive/MyDrive/what-is-the-book-2/runs")
assert (PROJECT / ".venv/bin/python").exists(), "Instale as dependencias primeiro."
assert (PROJECT / "data/unified/train/images").exists(), "Extraia o dataset primeiro."
assert Path("/content/drive/MyDrive").exists(), "Monte o Google Drive primeiro."
os.chdir(PROJECT)
env = {**os.environ, "MPLBACKEND": "Agg", "PYTHONUNBUFFERED": "1"}
logs = OUTPUT / "logs"
logs.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = logs / f"retomada_automatica_{stamp}.log"
print(f"Log completo: {log_path}", flush=True)

with log_path.open("w", encoding="utf-8") as log:
    process = subprocess.Popen(
        ["uv", "run", "--no-sync", "python", "-u", "-c", code],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    ultima_epoca = None
    try:
        for line in process.stdout:
            log.write(line)
            log.flush()
            parts = line.strip().split()
            token = parts[0] if parts else ""
            epoch_parts = token.split("/")
            is_epoch = len(epoch_parts) == 2 and all(part.isdigit() for part in epoch_parts)
            if "STATUS |" in line:
                ultima_epoca = None
                print(line.rstrip(), flush=True)
            elif is_epoch and token != ultima_epoca:
                ultima_epoca = token
                print(f"Epoca {token}", flush=True)
            elif any(
                term in line
                for term in (
                    "Traceback",
                    "Error",
                    "WARNING",
                    "epochs completed",
                    "Results saved",
                )
            ):
                print(line.rstrip(), flush=True)
        returncode = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        print("Processo interrompido. Checkpoints ja salvos foram preservados.")
        raise

print("Codigo de saida:", returncode)
if returncode != 0:
    print("Ultimas linhas do log:")
    print("\n".join(log_path.read_text().splitlines()[-40:]))
    raise RuntimeError(f"Treinamento interrompido com erro. Consulte {log_path}")